In [7]:
# 📌 Inference + Metrics Export (publish-ready)
import os, cv2, json, torch, timm
import pandas as pd
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# ===== PATHS =====
TEST_DIR = "/home/gpu/PSL Isolated Signs Datasets/PSL-Dataset-2021/train"
CKPT     = "/home/gpu/PSL Isolated Signs Datasets/PSL-Images-2021-Dataset-out/best_deit3_PSL-Images-21-out.pth"
OUT_DIR  = "/home/gpu/PSL Isolated Signs Datasets/Results-PSL-Images-OUT-VIT/VIT-Resutls-30-8-25"
os.makedirs(OUT_DIR, exist_ok=True)

# ===== UTILS =====
def safe_imread_rgb(path: str):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise RuntimeError(f"Failed to load image: {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

class AlbumentationsImageFolder(ImageFolder):
    def __init__(self, root, transform):
        super().__init__(root=root); self.atransform = transform
    def __getitem__(self, idx):
        path, target = self.samples[idx]
        img = safe_imread_rgb(path)
        img = self.atransform(image=img)["image"]
        return img, target

def make_val_tfms(img_size, mean, std):
    return A.Compose([
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(min_height=img_size, min_width=img_size, border_mode=cv2.BORDER_REFLECT_101),
        A.Normalize(mean=tuple(mean), std=tuple(std)),
        ToTensorV2(),
    ])

@torch.no_grad()
def evaluate(model, loader, device, num_classes):
    crit = torch.nn.CrossEntropyLoss()
    y_true, y_pred, total, loss_meter, top5_correct = [], [], 0, 0.0, 0
    k = min(5, num_classes)

    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        logits = model(images)
        loss = crit(logits, targets)

        probs = torch.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)

        y_true.extend(targets.cpu().numpy()); y_pred.extend(preds.cpu().numpy())
        loss_meter += loss.item() * images.size(0); total += images.size(0)

        topk_idx = torch.topk(probs, k=k, dim=1).indices
        top5_correct += topk_idx.eq(targets.view(-1,1)).any(dim=1).sum().item()

    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="macro", zero_division=0)
    cm   = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    report = classification_report(y_true, y_pred, labels=list(range(num_classes)), output_dict=True)

    return {
        "acc": acc*100, "prec": prec*100, "rec": rec*100, "f1": f1*100,
        "top1": acc*100, "top5": 100*top5_correct/max(1,total),
        "loss": loss_meter/max(1,total), "cm": cm,
        "report": report
    }

# ===== MAIN =====
device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt = torch.load(CKPT, map_location=device)
model_name  = ckpt["model_name"]; num_classes = ckpt["classes"]
class_names = ckpt["class_names"]; cfg = ckpt["cfg"]
img_size = ckpt["img_size"]; mean, std = cfg["mean"], cfg["std"]

# Build + load model
model = timm.create_model(model_name, pretrained=False, num_classes=num_classes).to(device)
model.load_state_dict(ckpt["model"], strict=True); model.eval()

# Dataset / loader
tfms = make_val_tfms(img_size, mean, std)
test_ds = AlbumentationsImageFolder(TEST_DIR, tfms)
assert test_ds.classes == class_names, "Class mismatch with checkpoint"
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=8, pin_memory=True)

# Evaluate
metrics = evaluate(model, test_loader, device, num_classes)

# ===== SAVE RESULTS =====
# 1. Human-readable summary
with open(os.path.join(OUT_DIR, "metrics.txt"), "w") as f:
    f.write(f"Model: {model_name}\n")
    f.write(f"Test Images: {len(test_ds)} | Classes: {num_classes}\n\n")
    f.write(f"Top-1 Acc: {metrics['top1']:.2f}%\n")
    f.write(f"Top-5 Acc: {metrics['top5']:.2f}%\n")
    f.write(f"Precision (macro): {metrics['prec']:.2f}%\n")
    f.write(f"Recall (macro): {metrics['rec']:.2f}%\n")
    f.write(f"F1 (macro): {metrics['f1']:.2f}%\n")
    f.write(f"CrossEntropy Loss: {metrics['loss']:.6f}\n")

# 2. JSON
json.dump(metrics, open(os.path.join(OUT_DIR,"metrics.json"),"w"), indent=2, default=lambda x:x.tolist())

# 3. CSVs
pd.DataFrame(metrics["cm"], index=class_names, columns=class_names)\
  .to_csv(os.path.join(OUT_DIR,"confusion_matrix.csv"))
rows=[]
for i,cls in enumerate(class_names):
    stats=metrics["report"][str(i)]
    rows.append({"class":cls, "precision":stats["precision"], "recall":stats["recall"],
                 "f1":stats["f1-score"], "support":stats["support"]})
pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR,"per_class.csv"), index=False)

# 4. Excel (summary + confusion matrix + per-class)
with pd.ExcelWriter(os.path.join(OUT_DIR,"metrics.xlsx"), engine="openpyxl") as writer:
    pd.DataFrame([{
        "Top1_%":metrics["top1"], "Top5_%":metrics["top5"],
        "Macro_Precision_%":metrics["prec"], "Macro_Recall_%":metrics["rec"],
        "Macro_F1_%":metrics["f1"], "Loss":metrics["loss"]
    }]).to_excel(writer, index=False, sheet_name="summary")
    pd.DataFrame(metrics["cm"], index=class_names, columns=class_names)\
        .to_excel(writer, sheet_name="confusion_matrix")
    pd.DataFrame(rows).to_excel(writer, index=False, sheet_name="per_class")

print(f"✅ Results saved to {OUT_DIR}")


/tmp/ipykernel_1258845/3544373413.py:79: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(CKPT, map_location=device)


FileNotFoundError: Found no valid file for the classes .ipynb_checkpoints. Supported extensions are: .jpg, .jpeg, .png, .ppm, .bmp, .pgm, .tif, .tiff, .webp

In [7]:
# 📌 Inference + Metrics Export (publish-ready)
import os, cv2, json, torch, timm
import pandas as pd
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# ===== PATHS =====
TEST_DIR = "/home/gpu/PSL Isolated Signs Datasets/PSL-Dataset-2021/train"
CKPT     = "/home/gpu/PSL Isolated Signs Datasets/PSL-Images-2021-Dataset-out/best_deit3_PSL-Images-21-out.pth"
OUT_DIR  = "/home/gpu/PSL Isolated Signs Datasets/Results-PSL-Images-OUT-VIT/VIT-Resutls-30-8-25"
os.makedirs(OUT_DIR, exist_ok=True)

# ===== UTILS =====
def safe_imread_rgb(path: str):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise RuntimeError(f"Failed to load image: {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

class AlbumentationsImageFolder(ImageFolder):
    def __init__(self, root, transform):
        super().__init__(root=root); self.atransform = transform
    def __getitem__(self, idx):
        path, target = self.samples[idx]
        img = safe_imread_rgb(path)
        img = self.atransform(image=img)["image"]
        return img, target

def make_val_tfms(img_size, mean, std):
    return A.Compose([
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(min_height=img_size, min_width=img_size, border_mode=cv2.BORDER_REFLECT_101),
        A.Normalize(mean=tuple(mean), std=tuple(std)),
        ToTensorV2(),
    ])

@torch.no_grad()
def evaluate(model, loader, device, num_classes):
    crit = torch.nn.CrossEntropyLoss()
    y_true, y_pred, total, loss_meter, top5_correct = [], [], 0, 0.0, 0
    k = min(5, num_classes)

    for images, targets in loader:
        images, targets = images.to(device), targets.to(device)
        logits = model(images)
        loss = crit(logits, targets)

        probs = torch.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)

        y_true.extend(targets.cpu().numpy()); y_pred.extend(preds.cpu().numpy())
        loss_meter += loss.item() * images.size(0); total += images.size(0)

        topk_idx = torch.topk(probs, k=k, dim=1).indices
        top5_correct += topk_idx.eq(targets.view(-1,1)).any(dim=1).sum().item()

    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="macro", zero_division=0)
    cm   = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    report = classification_report(y_true, y_pred, labels=list(range(num_classes)), output_dict=True)

    return {
        "acc": acc*100, "prec": prec*100, "rec": rec*100, "f1": f1*100,
        "top1": acc*100, "top5": 100*top5_correct/max(1,total),
        "loss": loss_meter/max(1,total), "cm": cm,
        "report": report
    }

# ===== MAIN =====
device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt = torch.load(CKPT, map_location=device)
model_name  = ckpt["model_name"]; num_classes = ckpt["classes"]
class_names = ckpt["class_names"]; cfg = ckpt["cfg"]
img_size = ckpt["img_size"]; mean, std = cfg["mean"], cfg["std"]

# Build + load model
model = timm.create_model(model_name, pretrained=False, num_classes=num_classes).to(device)
model.load_state_dict(ckpt["model"], strict=True); model.eval()

# Dataset / loader
tfms = make_val_tfms(img_size, mean, std)
test_ds = AlbumentationsImageFolder(TEST_DIR, tfms)
assert test_ds.classes == class_names, "Class mismatch with checkpoint"
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=8, pin_memory=True)

# Evaluate
metrics = evaluate(model, test_loader, device, num_classes)

# ===== SAVE RESULTS =====
# 1. Human-readable summary
with open(os.path.join(OUT_DIR, "metrics.txt"), "w") as f:
    f.write(f"Model: {model_name}\n")
    f.write(f"Test Images: {len(test_ds)} | Classes: {num_classes}\n\n")
    f.write(f"Top-1 Acc: {metrics['top1']:.2f}%\n")
    f.write(f"Top-5 Acc: {metrics['top5']:.2f}%\n")
    f.write(f"Precision (macro): {metrics['prec']:.2f}%\n")
    f.write(f"Recall (macro): {metrics['rec']:.2f}%\n")
    f.write(f"F1 (macro): {metrics['f1']:.2f}%\n")
    f.write(f"CrossEntropy Loss: {metrics['loss']:.6f}\n")

# 2. JSON
json.dump(metrics, open(os.path.join(OUT_DIR,"metrics.json"),"w"), indent=2, default=lambda x:x.tolist())

# 3. CSVs
pd.DataFrame(metrics["cm"], index=class_names, columns=class_names)\
  .to_csv(os.path.join(OUT_DIR,"confusion_matrix.csv"))
rows=[]
for i,cls in enumerate(class_names):
    stats=metrics["report"][str(i)]
    rows.append({"class":cls, "precision":stats["precision"], "recall":stats["recall"],
                 "f1":stats["f1-score"], "support":stats["support"]})
pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR,"per_class.csv"), index=False)

# 4. Excel (summary + confusion matrix + per-class)
with pd.ExcelWriter(os.path.join(OUT_DIR,"metrics.xlsx"), engine="openpyxl") as writer:
    pd.DataFrame([{
        "Top1_%":metrics["top1"], "Top5_%":metrics["top5"],
        "Macro_Precision_%":metrics["prec"], "Macro_Recall_%":metrics["rec"],
        "Macro_F1_%":metrics["f1"], "Loss":metrics["loss"]
    }]).to_excel(writer, index=False, sheet_name="summary")
    pd.DataFrame(metrics["cm"], index=class_names, columns=class_names)\
        .to_excel(writer, sheet_name="confusion_matrix")
    pd.DataFrame(rows).to_excel(writer, index=False, sheet_name="per_class")

print(f"✅ Results saved to {OUT_DIR}")


/tmp/ipykernel_1258845/3544373413.py:79: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(CKPT, map_location=device)


FileNotFoundError: Found no valid file for the classes .ipynb_checkpoints. Supported extensions are: .jpg, .jpeg, .png, .ppm, .bmp, .pgm, .tif, .tiff, .webp

In [1]:
# All-in-one TTA eval & inference (Albumentations 2.0.8)

import os, csv, argparse
from pathlib import Path
from typing import List, Tuple

import cv2, numpy as np, torch, torch.nn as nn, timm
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import ImageFolder
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ---------- Utils ----------
def safe_imread_rgb(path: str):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None: 
        raise RuntimeError(f"Failed to read: {path}")
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    elif img.ndim == 3:
        if img.shape[2] == 4:
            img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    else:
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

def load_checkpoint(weights_path: str):
    ckpt = torch.load(weights_path, map_location="cpu")
    model_name = ckpt.get("model_name", "deit3_base_patch16_224")
    num_classes = ckpt.get("classes", 39)
    img_size = ckpt.get("img_size", 224)
    state_dict = ckpt["model"]
    return model_name, num_classes, img_size, state_dict

def build_model(model_name: str, num_classes: int, state_dict: dict, device: str):
    model = timm.create_model(model_name, pretrained=False, num_classes=num_classes)
    model.load_state_dict(state_dict, strict=True)
    model.to(device).eval()
    return model

# ---------- TTA builders ----------
def make_single_tta(img_size: int, scale: float, angle: float, do_flip: bool):
    ops = []
    # 1) Scale longest side
    max_side = max(1, int(round(img_size * scale)))
    ops.append(A.LongestMaxSize(max_size=max_side, interpolation=cv2.INTER_CUBIC))
    # 2) Guarantee both dims >= img_size (prevents CropSizeError)
    ops.append(A.PadIfNeeded(min_height=img_size, min_width=img_size,
                             border_mode=cv2.BORDER_REFLECT_101))
    # 3) Small rotation without changing canvas size
    if abs(angle) > 0:
        ops.append(A.Affine(rotate=angle, fit_output=False,
                            border_mode=cv2.BORDER_REFLECT_101))
    # 4) Optional flip (enable only if left/right isn't a distinct class)
    if do_flip:
        ops.append(A.HorizontalFlip(p=1.0))
    # 5) Finalize exact size + normalize
    ops.append(A.CenterCrop(img_size, img_size))
    ops += [
        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ]
    return A.Compose(ops)

def make_tta_list(img_size: int, scales: List[float], angles: List[float], flip: bool):
    tfs = []
    for s in scales:
        for a in angles:
            tfs.append(make_single_tta(img_size, s, a, False))
            if flip:
                tfs.append(make_single_tta(img_size, s, a, True))
    return tfs

# ---------- Datasets (raw RGB -> transform per TTA pass) ----------
class AlbumentationsImageFolderRaw(ImageFolder):
    def __init__(self, root):
        super().__init__(root=root)
    def __getitem__(self, idx):
        path, target = self.samples[idx]
        img = safe_imread_rgb(path)
        return img, target

class ImageListRaw(Dataset):
    def __init__(self, files: List[str]):
        self.files = files
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        p = self.files[idx]
        img = safe_imread_rgb(p)
        return img, p

def list_images(path: str) -> List[str]:
    exts = {".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}
    p = Path(path)
    if p.is_file(): return [str(p)]
    return sorted([str(f) for f in p.rglob("*") if f.suffix.lower() in exts])

# ---------- Core TTA loops ----------
import csv, numpy as np, torch
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import ImageFolder

@torch.no_grad()
def eval_with_tta(data_root: str, weights: str, out_dir: str,
                  scales=[0.9,1.0,1.1], angles=[-8,0,8], flip=False,
                  bs=64, workers=0):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # --- load model ---
    ckpt = torch.load(weights, map_location="cpu")
    model_name = ckpt.get("model_name", "deit3_base_patch16_224")
    n_cls_ckpt = int(ckpt.get("classes", 39))
    img_size = int(ckpt.get("img_size", 224))
    state = ckpt["model"]

    model = timm.create_model(model_name, pretrained=False, num_classes=n_cls_ckpt)
    model.load_state_dict(state, strict=True)
    model.to(device).eval()

    # --- dataset (raw RGB; we apply tfms per TTA pass) ---
    class AlbumentationsImageFolderRaw(ImageFolder):
        def __init__(self, root): super().__init__(root=root)
        def __getitem__(self, idx):
            path, y = self.samples[idx]
            return safe_imread_rgb(path), y

    base_ds = AlbumentationsImageFolderRaw(data_root)
    classes_ds = base_ds.classes
    n_cls_ds = len(classes_ds)
    N = len(base_ds)

    print(f"[INFO] ckpt_classes={n_cls_ckpt} | dataset_classes={n_cls_ds} | N={N}")

    # --- TTA builders (use your patched make_single_tta) ---
    tta_list = make_tta_list(img_size, scales, angles, flip)

    # we’ll hold logits summed across TTA passes (shape uses ckpt class count)
    sum_logits = torch.zeros((N, n_cls_ckpt), dtype=torch.float32, device=device)
    use_amp = torch.cuda.is_available()

    class _View(Dataset):
        def __init__(self, base, tfm): self.base, self.tfm = base, tfm
        def __len__(self): return len(self.base)
        def __getitem__(self, i):
            img, y = self.base[i]
            x = self.tfm(image=img)["image"]
            return x, y

    for t_idx, tfm in enumerate(tta_list, 1):
        view = _View(base_ds, tfm)
        dl = DataLoader(view, batch_size=bs, shuffle=False, num_workers=workers, pin_memory=True)
        ofs = 0
        for xb, yb in dl:
            xb = xb.to(device, non_blocking=True)
            if use_amp:
                with torch.cuda.amp.autocast():
                    logits = model(xb)
            else:
                logits = model(xb)
            B = xb.size(0)
            sum_logits[ofs:ofs+B] += logits
            ofs += B
        print(f"TTA pass {t_idx}/{len(tta_list)} complete.")

    preds = sum_logits.argmax(dim=1).cpu().numpy()
    targets = np.array([y for _, y in base_ds.samples], dtype=np.int64)
    acc = (preds == targets).mean() * 100.0

    # --- robust confusion matrix & reports ---
    K = max(n_cls_ckpt, n_cls_ds)
    cm = np.zeros((K, K), dtype=np.int64)
    for t, p in zip(targets, preds):
        if 0 <= t < K and 0 <= p < K:
            cm[t, p] += 1

    # names padded to K so CSV writing never overflows
    names = list(classes_ds)
    if len(names) < K:
        names += [f"class_{i}" for i in range(len(names), K)]

    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, "eval_tta_summary.txt"), "w", encoding="utf-8") as f:
        f.write(f"Model: {model_name}\n")
        f.write(f"ckpt_classes: {n_cls_ckpt} | dataset_classes: {n_cls_ds}\n")
        f.write(f"Image size: {img_size}\n")
        f.write(f"TTA: scales={scales}, angles={angles}, flip={flip}\n")
        f.write(f"Accuracy: {acc:.2f}%\n")

    # per-class accuracy only for existing dataset classes
    with open(os.path.join(out_dir, "per_class_acc.txt"), "w", encoding="utf-8") as f:
        for i in range(n_cls_ds):
            total_i = cm[i, :].sum()
            pc = 100.0 * (cm[i, i] / total_i) if total_i > 0 else 0.0
            f.write(f"{names[i]}: {pc:.2f}\n")

    with open(os.path.join(out_dir, "confusion_matrix_tta.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow([""] + names[:K])
        for i in range(K):
            w.writerow([names[i]] + cm[i].tolist())

    print(f"TTA eval done. Acc: {acc:.2f}% → {out_dir}")
    return acc

@torch.no_grad()
def infer_with_tta(data_path: str, weights: str, out_csv: str,
                   label_names_root="", scales=[0.9,1.0,1.1], angles=[-8,0,8], flip=False,
                   bs=64, workers=4):
    """Infer on unlabeled images; save CSV with top1 + prob (TTA averaged)."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model_name, n_cls, img_size, state = load_checkpoint(weights)
    model = build_model(model_name, n_cls, state, device)

    files = list_images(data_path)
    if not files:
        print(f"No images at {data_path}"); 
        return

    # class names (optional)
    if label_names_root and Path(label_names_root).is_dir():
        tmp = ImageFolder(label_names_root)
        class_names = tmp.classes
    else:
        class_names = [f"class_{i}" for i in range(n_cls)]

    base_ds = ImageListRaw(files)
    N = len(base_ds)
    tta_list = make_tta_list(img_size, scales, angles, flip)
    sum_logits = torch.zeros((N, n_cls), dtype=torch.float32, device=device)

    use_amp = torch.cuda.is_available()
    for t_idx, tfm in enumerate(tta_list, 1):
        class _View(Dataset):
            def __init__(self, base, tfm): self.base, self.tfm = base, tfm
            def __len__(self): return len(self.base)
            def __getitem__(self, i):
                img, p = self.base[i]
                x = self.tfm(image=img)["image"]
                return x, p
        view = _View(base_ds, tfm)
        dl = DataLoader(view, batch_size=bs, shuffle=False, num_workers=workers, pin_memory=True)
        ofs = 0
        for xb, paths in dl:
            xb = xb.to(device, non_blocking=True)
            if use_amp:
                with torch.cuda.amp.autocast():
                    logits = model(xb)
            else:
                logits = model(xb)
            B = xb.size(0)
            sum_logits[ofs:ofs+B] += logits
            ofs += B
        print(f"TTA pass {t_idx}/{len(tta_list)} complete.")

    probs = sum_logits.softmax(dim=1).cpu().numpy()
    top1_idx = probs.argmax(axis=1)
    top1_prob = probs.max(axis=1)

    os.makedirs(os.path.dirname(out_csv) or ".", exist_ok=True)
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["path","top1","top1_prob"])
        for p, cls_i, pr in zip(files, top1_idx, top1_prob):
            w.writerow([p, class_names[int(cls_i)], f"{pr:.4f}"])
    print(f"Inference (TTA) saved → {out_csv}")


/home/gpu/.local/lib/python3.12/site-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.24). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [7]:
# Paths from your training run
WEIGHTS = "/home/gpu/PSL Isolated Signs Datasets/best_vit_psl.pth"
LABELED_DATA = "/home/gpu/PSL-Cropped/val"
OUT_DIR = "/home/gpu/PSL Isolated Signs Datasets/eval_tta_out"

# If left/right hand is NOT class-specific, set flip=True for a small boost
FLIP_SAFE = False   # ← change to True if safe for your labels

# Highest-accuracy evaluation with TTA
acc = eval_with_tta(
    data_root=LABELED_DATA,
    weights=WEIGHTS,
    out_dir=OUT_DIR,
    scales=[0.9, 1.0, 1.1],
    angles=[-8, 0, 8],
    flip=FLIP_SAFE,
    bs=64,
    workers=4
)
print("Final TTA Accuracy:", f"{acc:.2f}%")


/tmp/ipykernel_197119/213459721.py:112: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(weights, map_location="cpu")


[INFO] ckpt_classes=39 | dataset_classes=38 | N=833


/tmp/ipykernel_197119/213459721.py:158: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


TTA pass 1/9 complete.
TTA pass 2/9 complete.
TTA pass 3/9 complete.
TTA pass 4/9 complete.
TTA pass 5/9 complete.
TTA pass 6/9 complete.
TTA pass 7/9 complete.
TTA pass 8/9 complete.
TTA pass 9/9 complete.
TTA eval done. Acc: 72.27% → /home/gpu/PSL Isolated Signs Datasets/eval_tta_out
Final TTA Accuracy: 72.27%


In [ ]:
# If not installed:
# %pip install -q scipy

import os, csv, numpy as np, torch, timm
from pathlib import Path
from typing import List
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
from albumentations.pytorch import ToTensorV2
import albumentations as A, cv2

from scipy.optimize import linear_sum_assignment  # Hungarian

def eval_with_tta_and_mapping(data_root: str, weights: str, out_dir: str,
                              scales=[0.9,1.0,1.1], angles=[-8,0,8], flip=False,
                              bs=64, workers=0):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # --- load model/ckpt meta ---
    ckpt = torch.load(weights, map_location="cpu")
    model_name = ckpt.get("model_name", "deit3_base_patch16_224")
    n_cls_ckpt = int(ckpt.get("classes", 39))
    img_size = int(ckpt.get("img_size", 224))
    state = ckpt["model"]

    model = timm.create_model(model_name, pretrained=False, num_classes=n_cls_ckpt)
    model.load_state_dict(state, strict=True)
    model.to(device).eval()

    # --- dataset (raw, tfm applied per TTA) ---
    class ImageFolderRaw(ImageFolder):
        def __init__(self, root): super().__init__(root=root)
        def __getitem__(self, idx):
            path, y = self.samples[idx]
            img = safe_imread_rgb(path)
            return img, y

    base_ds = ImageFolderRaw(data_root)
    classes_ds = base_ds.classes
    n_cls_ds = len(classes_ds)
    N = len(base_ds)

    print(f"[INFO] ckpt_classes={n_cls_ckpt} | dataset_classes={n_cls_ds} | N={N}")

    # --- TTA builders (uses your patched make_single_tta) ---
    tta_list = make_tta_list(img_size, scales, angles, flip)

    # accumulate logits over TTAs
    sum_logits = torch.zeros((N, n_cls_ckpt), dtype=torch.float32, device=device)
    use_amp = torch.cuda.is_available()

    class _View(Dataset):
        def __init__(self, base, tfm): self.base, self.tfm = base, tfm
        def __len__(self): return len(self.base)
        def __getitem__(self, i):
            img, y = self.base[i]
            x = self.tfm(image=img)["image"]
            return x, y

    for t_idx, tfm in enumerate(tta_list, 1):
        view = _View(base_ds, tfm)
        dl = DataLoader(view, batch_size=bs, shuffle=False, num_workers=workers, pin_memory=True)
        ofs = 0
        for xb, yb in dl:
            xb = xb.to(device, non_blocking=True)
            if use_amp:
                with torch.amp.autocast("cuda"):
                    logits = model(xb)
            else:
                logits = model(xb)
            B = xb.size(0)
            sum_logits[ofs:ofs+B] += logits
            ofs += B
        print(f"TTA pass {t_idx}/{len(tta_list)} complete.")

    # raw predictions vs dataset labels (indices as-is)
    preds_raw = sum_logits.argmax(1).cpu().numpy()
    targets = np.array([y for _, y in base_ds.samples], dtype=np.int64)

    # --- build rectangular confusion (dataset classes x ckpt classes) ---
    cm_rect = np.zeros((n_cls_ds, n_cls_ckpt), dtype=np.int64)
    for t, p in zip(targets, preds_raw):
        cm_rect[t, p] += 1

    # --- find optimal mapping dataset_class -> ckpt_class ---
    # maximize matches => minimize negative counts
    row_ind, col_ind = linear_sum_assignment(cm_rect.max() - cm_rect)
    # build inverse mapping: ckpt_class -> dataset_class (for remapping preds)
    inv_map = {int(c): int(r) for r, c in zip(row_ind, col_ind)}  # only mapped ones

    # remap predictions to dataset label space
    preds_remap = np.array([inv_map.get(int(p), -1) for p in preds_raw], dtype=np.int64)

    # compute accuracies
    acc_raw = (preds_raw == targets).mean() * 100.0
    mask = preds_remap >= 0
    acc_mapped = (preds_remap[mask] == targets[mask]).mean() * 100.0 if mask.any() else 0.0

    # save mapping + reports
    os.makedirs(out_dir, exist_ok=True)
    map_csv = os.path.join(out_dir, "label_mapping_ckpt_to_dataset.csv")
    with open(map_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["dataset_class_idx","dataset_class_name","ckpt_class_idx","hits"])
        for r, c in zip(row_ind, col_ind):
            w.writerow([r, classes_ds[r], c, cm_rect[r, c]])
    with open(os.path.join(out_dir, "eval_tta_summary_mapped.txt"), "w", encoding="utf-8") as f:
        f.write(f"Raw acc (no mapping): {acc_raw:.2f}%\n")
        f.write(f"Mapped acc (Hungarian): {acc_mapped:.2f}%\n")
        f.write(f"ckpt_classes={n_cls_ckpt} | dataset_classes={n_cls_ds} | N={N}\n")
        f.write(f"TTA scales={scales}, angles={angles}, flip={flip}\n")

    print(f"Raw acc: {acc_raw:.2f}%  |  Mapped acc: {acc_mapped:.2f}%")
    print(f"Saved mapping → {map_csv}")
    return acc_raw, acc_mapped

# === RUN ===
WEIGHTS = "/home/gpu/PSL Isolated Signs Datasets/best_vit_psl.pth"
LABELED_DATA = "/home/gpu/PSL Isolated Signs Datasets/Combined-cropped-dataset"
OUT_DIR = "/home/gpu/PSL Isolated Signs Datasets/eval_tta_out"

# set flip=True ONLY if left/right hand is NOT a separate label
acc_raw, acc_mapped = eval_with_tta_and_mapping(
    data_root=LABELED_DATA,
    weights=WEIGHTS,
    out_dir=OUT_DIR,
    scales=[0.9, 1.0, 1.1],
    angles=[-8, 0, 8],
    flip=False,
    bs=64, workers=0
)
print("Final (mapped) TTA Accuracy:", f"{acc_mapped:.2f}%")


In [1]:
# === All-in-one TTA eval with label-mapping fix (Jupyter-friendly) ===
# Requires: albumentations==2.x, timm, torch, torchvision, opencv-python
# Optional (better mapping): scipy  ->  %pip install -q scipy

import os, csv
from pathlib import Path
from typing import List

import numpy as np
import torch, torch.nn as nn, timm
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder

import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ---------- Utils ----------
def safe_imread_rgb(path: str):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise RuntimeError(f"Failed to read: {path}")
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    elif img.ndim == 3:
        if img.shape[2] == 4:
            img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    else:
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

# ---------- TTA builders (pad BEFORE crop to avoid CropSizeError) ----------
def make_single_tta(img_size: int, scale: float, angle: float, do_flip: bool):
    ops = []
    max_side = max(1, int(round(img_size * scale)))
    ops.append(A.LongestMaxSize(max_size=max_side, interpolation=cv2.INTER_CUBIC))
    ops.append(A.PadIfNeeded(min_height=img_size, min_width=img_size,
                             border_mode=cv2.BORDER_REFLECT_101))
    if abs(angle) > 0:
        ops.append(A.Affine(rotate=angle, fit_output=False,
                            border_mode=cv2.BORDER_REFLECT_101))
    if do_flip:
        ops.append(A.HorizontalFlip(p=1.0))
    ops += [
        A.CenterCrop(img_size, img_size),
        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ]
    return A.Compose(ops)

def make_tta_list(img_size: int, scales: List[float], angles: List[float], flip: bool):
    tfs = []
    for s in scales:
        for a in angles:
            tfs.append(make_single_tta(img_size, s, a, False))
            if flip:
                tfs.append(make_single_tta(img_size, s, a, True))
    return tfs

# ---------- Eval with TTA + optimal label mapping ----------
@torch.no_grad()
def eval_with_tta_and_mapping(
    data_root: str, weights: str, out_dir: str,
    scales=[0.9, 1.0, 1.1], angles=[-8, 0, 8], flip=False,
    bs=64, workers=0
):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Load checkpoint & model
    ckpt = torch.load(weights, map_location="cpu")
    model_name = ckpt.get("model_name", "deit3_base_patch16_224")
    n_cls_ckpt = int(ckpt.get("classes", 39))
    img_size = int(ckpt.get("img_size", 224))
    state = ckpt["model"]

    model = timm.create_model(model_name, pretrained=False, num_classes=n_cls_ckpt)
    model.load_state_dict(state, strict=True)
    model.to(device).eval()

    # Raw dataset (we'll apply TTA transforms per pass)
    class ImageFolderRaw(ImageFolder):
        def __init__(self, root): super().__init__(root=root)
        def __getitem__(self, idx):
            path, y = self.samples[idx]
            return safe_imread_rgb(path), y

    base_ds = ImageFolderRaw(data_root)
    classes_ds = base_ds.classes
    n_cls_ds = len(classes_ds)
    N = len(base_ds)

    print(f"[INFO] ckpt_classes={n_cls_ckpt} | dataset_classes={n_cls_ds} | N={N}")

    tta_list = make_tta_list(img_size, scales, angles, flip)
    sum_logits = torch.zeros((N, n_cls_ckpt), dtype=torch.float32, device=device)

    class _View(Dataset):
        def __init__(self, base, tfm): self.base, self.tfm = base, tfm
        def __len__(self): return len(self.base)
        def __getitem__(self, i):
            img, y = self.base[i]
            x = self.tfm(image=img)["image"]
            return x, y

    use_amp = torch.cuda.is_available()
    for t_idx, tfm in enumerate(tta_list, 1):
        view = _View(base_ds, tfm)
        dl = DataLoader(view, batch_size=bs, shuffle=False, num_workers=workers, pin_memory=True)
        ofs = 0
        for xb, yb in dl:
            xb = xb.to(device, non_blocking=True)
            if use_amp:
                with torch.amp.autocast("cuda"):
                    logits = model(xb)
            else:
                logits = model(xb)
            B = xb.size(0)
            sum_logits[ofs:ofs+B] += logits
            ofs += B
        print(f"TTA pass {t_idx}/{len(tta_list)} complete.")

    preds_raw = sum_logits.argmax(1).cpu().numpy()
    targets = np.array([y for _, y in base_ds.samples], dtype=np.int64)

    # Rectangular confusion: dataset classes x ckpt classes
    cm_rect = np.zeros((n_cls_ds, n_cls_ckpt), dtype=np.int64)
    for t, p in zip(targets, preds_raw):
        cm_rect[t, p] += 1

    # Hungarian (optimal) mapping dataset_class -> ckpt_class
    try:
        from scipy.optimize import linear_sum_assignment
        row_ind, col_ind = linear_sum_assignment(cm_rect.max() - cm_rect)
    except Exception as e:
        # Fallback greedy mapping (if scipy not installed)
        row_ind, col_ind = [], []
        used = set()
        for r in range(n_cls_ds):
            order = np.argsort(-cm_rect[r])  # descending
            c_pick = next((c for c in order if c not in used), int(order[0]))
            row_ind.append(r); col_ind.append(c_pick); used.add(c_pick)
        row_ind, col_ind = np.array(row_ind), np.array(col_ind)
        print("[WARN] SciPy not available; used greedy mapping. Install scipy for optimal mapping.")

    inv_map = {int(c): int(r) for r, c in zip(row_ind, col_ind)}  # ckpt_idx -> dataset_idx
    preds_mapped = np.array([inv_map.get(int(p), -1) for p in preds_raw], dtype=np.int64)

    acc_raw = (preds_raw == targets).mean() * 100.0
    mask = preds_mapped >= 0
    acc_mapped = (preds_mapped[mask] == targets[mask]).mean() * 100.0 if mask.any() else 0.0

    # Save reports
    os.makedirs(out_dir, exist_ok=True)
    map_csv = os.path.join(out_dir, "label_mapping_ckpt_to_dataset.csv")
    with open(map_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["dataset_class_idx","dataset_class_name","ckpt_class_idx","hits"])
        for r, c in zip(row_ind, col_ind):
            w.writerow([r, classes_ds[r] if r < len(classes_ds) else f"class_{r}", c, cm_rect[r, c]])

    with open(os.path.join(out_dir, "eval_tta_summary_mapped.txt"), "w", encoding="utf-8") as f:
        f.write(f"Raw acc (no mapping): {acc_raw:.2f}%\n")
        f.write(f"Mapped acc (Hungarian/greedy): {acc_mapped:.2f}%\n")
        f.write(f"ckpt_classes={n_cls_ckpt} | dataset_classes={n_cls_ds} | N={N}\n")
        f.write(f"TTA scales={scales}, angles={angles}, flip={flip}\n")

    # Confusion matrix (mapped) limited to dataset classes
    K = n_cls_ds
    cm_mapped = np.zeros((K, K), dtype=np.int64)
    for t, p in zip(targets, preds_mapped):
        if 0 <= t < K and 0 <= p < K:
            cm_mapped[t, p] += 1
    with open(os.path.join(out_dir, "CFRT-LODO-confusion_matrix_tta_mapped.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f); w.writerow([""] + classes_ds)
        for i in range(K): w.writerow([classes_ds[i]] + cm_mapped[i].tolist())

    print(f"Raw acc: {acc_raw:.2f}%  |  Mapped acc: {acc_mapped:.2f}%")
    print(f"Saved mapping → {map_csv}")
    return acc_raw, acc_mapped

# ----------------- RUN -----------------
WEIGHTS = "/home/gpu/PSL Isolated Signs Datasets/CFRT-Dataset-out/best_deit3_CFRT-Dataset-out.pth"
LABELED_DATA = "/home/gpu/PSL-Cropped/train"
#LABELED_DATA = "/home/gpu/PSL Isolated Signs Datasets/Combined-cropped-dataset"
OUT_DIR = "/home/gpu/PSL Isolated Signs Datasets/Results-TTA-CFRT-LODO"

# Flip only if left/right hand is NOT label-specific
FLIP_SAFE = True

acc_raw, acc_mapped = eval_with_tta_and_mapping(
    data_root=LABELED_DATA,
    weights=WEIGHTS,
    out_dir=OUT_DIR,
    scales=[0.9, 1.0, 1.1],
    angles=[-8, 0, 8],
    flip=FLIP_SAFE,
    bs=64,
    workers=0  # JupyterHub-stable
)
print("Final (mapped) TTA Accuracy:", f"{acc_mapped:.2f}%")


/home/gpu/.local/lib/python3.12/site-packages/albumentations/__init__.py:24: UserWarning: A new version of Albumentations is available: 2.0.8 (you have 1.4.24). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()
/tmp/ipykernel_1258845/2336942651.py:72: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.se

[INFO] ckpt_classes=39 | dataset_classes=39 | N=1931
TTA pass 1/18 complete.
TTA pass 2/18 complete.
TTA pass 3/18 complete.
TTA pass 4/18 complete.
TTA pass 5/18 complete.
TTA pass 6/18 complete.
TTA pass 7/18 complete.
TTA pass 8/18 complete.
TTA pass 9/18 complete.
TTA pass 10/18 complete.
TTA pass 11/18 complete.
TTA pass 12/18 complete.
TTA pass 13/18 complete.
TTA pass 14/18 complete.
TTA pass 15/18 complete.
TTA pass 16/18 complete.
TTA pass 17/18 complete.
TTA pass 18/18 complete.
Raw acc: 39.10%  |  Mapped acc: 43.97%
Saved mapping → /home/gpu/PSL Isolated Signs Datasets/Results-TTA-CFRT-LODO/label_mapping_ckpt_to_dataset.csv
Final (mapped) TTA Accuracy: 43.97%


In [5]:
# 📌 Inference + Metrics Export (robust, TTA-capable, reproducible)
# - Handles: state_dict prefixes, class-order mismatches, dtype/device shapes
# - TTA levels: fast(4 views), strong(8), max(16). Identity view always included.
# - No flips (handedness can invert meaning in fingerspelling)
# - Exports: metrics.txt / metrics.json / confusion_matrix.csv / per_class.csv / metrics.xlsx

import os, cv2, json, math, time, torch, timm, random
import numpy as np
import pandas as pd
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# ===== USER CONFIG =====
TEST_DIR = "/home/gpu/PSL-Cropped/train"
CKPT     = "/home/gpu/PSL Isolated Signs Datasets/CFRT-Dataset-out/best_deit3_CFRT-Dataset-out.pth"
OUT_DIR  = "/home/gpu/PSL Isolated Signs Datasets/TTA-Results-CFRT-LODO/TTA-strong-VIT-Resutls-29-8-25"
TTA_LEVEL = "strong"       # {'fast','strong','max'}  (set to None or '' to disable TTA)
BATCH_NO_TTA = 128      # effective batch when TTA is off
BATCH_TTA    = 64       # base batch when TTA is on (each step does n_views*BATCH_TTA forward elems)

# ===== DEFAULTS / REPRO =====
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
RNG_SEED = 42  # deterministic where it makes sense (TTA remains stochastic by design)

# ===== UTILS =====
def set_seed(seed=RNG_SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False  # allow autotune for speed
    torch.backends.cudnn.benchmark = True

def safe_imread_rgb(path: str):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise RuntimeError(f"Failed to load image: {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

class AlbumentationsImageFolder(ImageFolder):
    """Returns tensor after deterministic val transforms."""
    def __init__(self, root, transform):
        super().__init__(root=root); self.atransform = transform
    def __getitem__(self, idx):
        path, target = self.samples[idx]
        img = safe_imread_rgb(path)
        img = self.atransform(image=img)["image"]
        return img, target

class NumpyImageFolder(ImageFolder):
    """For TTA: returns raw RGB numpy image (no transform)."""
    def __init__(self, root):
        super().__init__(root=root)
    def __getitem__(self, idx):
        path, target = self.samples[idx]
        img = safe_imread_rgb(path)  # HWC, uint8
        return img, target

def collate_numpy(batch):
    imgs, targets = zip(*batch)
    return list(imgs), torch.as_tensor(targets, dtype=torch.long)

def make_val_tfms(img_size, mean, std):
    return A.Compose([
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(min_height=img_size, min_width=img_size, border_mode=cv2.BORDER_REFLECT_101),
        A.Normalize(mean=tuple(mean), std=tuple(std)),
        ToTensorV2(),
    ])

# ---- Version-robust transform getter (handles renames/missing ops) ----
def maybe_op(name, **kwargs):
    """
    Return an albumentations transform if available; otherwise a safe fallback.
    Handles common version diffs (ImageCompression/JpegCompression, Defocus, Downscale).
    Unknown kwargs are filtered to match the available transform's signature.
    """
    T = getattr(A, name, None)
    if T is None:
        # Aliases / common renames
        if name in ("JpegCompression", "ImageCompression"):
            T = getattr(A, "ImageCompression", None) or getattr(A, "JpegCompression", None)
        elif name == "Defocus":
            return A.GaussianBlur(blur_limit=(3, 5), p=kwargs.get("p", 0.0))  # fallback
        elif name == "Downscale":
            return A.NoOp(p=0.0)
        else:
            return A.NoOp(p=0.0)

    # Filter kwargs to match signature (handles version changes)
    try:
        return T(**kwargs)
    except TypeError:
        from inspect import signature
        sig = signature(T)
        filtered = {k: v for k, v in kwargs.items() if k in sig.parameters}
        return T(**filtered)

def make_tta_aug(level, img_size, mean, std):
    """Return (n_views, aug) for stochastic TTA. Avoid flips (handedness). Version-robust."""
    base_ops = [
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(min_height=img_size, min_width=img_size, border_mode=cv2.BORDER_REFLECT_101),
    ]

    if level == "fast":
        n_views = 4
        ops = base_ops + [
            A.ShiftScaleRotate(shift_limit=0.02, scale_limit=0.05, rotate_limit=7,
                               border_mode=cv2.BORDER_REFLECT_101, p=0.8),
            A.RandomBrightnessContrast(brightness_limit=0.10, contrast_limit=0.10, p=0.5),
            A.GaussianBlur(blur_limit=(3,5), p=0.3),
            A.GaussNoise(var_limit=(5.0, 15.0), p=0.3),
        ]
    elif level == "strong":
        n_views = 8
        ops = base_ops + [
            A.ShiftScaleRotate(shift_limit=0.04, scale_limit=0.08, rotate_limit=12,
                               border_mode=cv2.BORDER_REFLECT_101, p=0.9),
            A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.7),
            A.HueSaturationValue(hue_shift_limit=5, sat_shift_limit=10, val_shift_limit=5, p=0.5),
            A.MotionBlur(blur_limit=5, p=0.3),
            A.GaussNoise(var_limit=(10.0, 25.0), p=0.4),
            maybe_op("ImageCompression", quality_lower=85, quality_upper=95, p=0.3),
        ]
    elif level == "max":
        n_views = 16
        ops = base_ops + [
            A.ShiftScaleRotate(shift_limit=0.06, scale_limit=0.10, rotate_limit=15,
                               border_mode=cv2.BORDER_REFLECT_101, p=1.0),
            A.RandomBrightnessContrast(brightness_limit=0.20, contrast_limit=0.20, p=0.8),
            A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=12, val_shift_limit=8, p=0.6),
            maybe_op("Defocus", radius=(2, 4), p=0.3),
            A.GaussNoise(var_limit=(15.0, 35.0), p=0.5),
            maybe_op("Downscale", scale_min=0.90, scale_max=0.95, p=0.3),
            maybe_op("ImageCompression", quality_lower=80, quality_upper=95, p=0.4),
        ]
    else:
        raise ValueError("TTA level must be one of {'fast','strong','max'}")

    ops += [A.Normalize(mean=tuple(mean), std=tuple(std)), ToTensorV2()]
    return n_views, A.Compose(ops)

def extract_model_info(ckpt_obj):
    """Robustly extract model info and state_dict from varied checkpoint formats."""
    sd = None
    model_name = ckpt_obj.get("model_name", None)
    num_classes = ckpt_obj.get("classes", None)
    class_names = ckpt_obj.get("class_names", None)
    cfg = ckpt_obj.get("cfg", {})
    img_size = ckpt_obj.get("img_size", None)

    # state dict can be under different keys
    for key in ["model", "state_dict", "model_state", "net", "ema_state_dict"]:
        if key in ckpt_obj and isinstance(ckpt_obj[key], dict):
            sd = ckpt_obj[key]
            break
    if sd is None:
        # raw state_dict?
        if all(isinstance(v, torch.Tensor) for v in ckpt_obj.values()):
            sd = ckpt_obj
        else:
            raise KeyError("No state_dict found in checkpoint.")

    # clean common prefixes
    def _clean_keys(d):
        new_d = {}
        for k, v in d.items():
            nk = k
            for pref in ["module.", "model.", "net.", "encoder.", "ema."]:
                if nk.startswith(pref):
                    nk = nk[len(pref):]
            new_d[nk] = v
        return new_d
    sd = _clean_keys(sd)

    # Fallbacks
    if class_names is not None:
        num_classes = len(class_names)
    if num_classes is None:
        raise ValueError("Checkpoint missing 'classes' or 'class_names'.")
    if model_name is None:
        raise ValueError("Checkpoint missing 'model_name' (needed to create model).")
    if img_size is None:
        img_size = 224
    mean = cfg.get("mean", IMAGENET_MEAN)
    std  = cfg.get("std", IMAGENET_STD)
    return sd, model_name, num_classes, class_names, img_size, mean, std

def build_label_remap(dataset_classes, ckpt_class_names):
    """
    Map dataset target indices (dataset_classes order) -> ckpt label indices (ckpt_class_names order).
    Ensures identical class sets; raises if mismatch.
    """
    if ckpt_class_names is None:
        # assume same order (legacy)
        return None
    set_ds = set(dataset_classes)
    set_ck = set(ckpt_class_names)
    if set_ds != set_ck:
        missing = set_ck - set_ds
        extra   = set_ds - set_ck
        raise ValueError(f"Class set mismatch.\nMissing in dataset: {sorted(list(missing))}\n"
                         f"Extra in dataset: {sorted(list(extra))}")
    remap = [-1]*len(dataset_classes)
    ck_index = {c:i for i,c in enumerate(ckpt_class_names)}
    for i, cls in enumerate(dataset_classes):
        remap[i] = ck_index[cls]
    remap = torch.tensor(remap, dtype=torch.long)
    return remap  # may be identity

@torch.inference_mode()
def evaluate_no_tta(model, loader, device, num_classes, label_remap=None, use_amp=True):
    crit = torch.nn.CrossEntropyLoss()
    y_true, y_pred, total, loss_meter, top5_correct = [], [], 0, 0.0, 0
    k = min(5, num_classes)
    amp_enabled = (use_amp and device.startswith("cuda"))
    for images, targets in loader:
        # remap targets to checkpoint label order (on CPU)
        if label_remap is not None:
            targets = label_remap[targets.cpu()].to(device, non_blocking=True)
        else:
            targets = targets.to(device, non_blocking=True)

        images = images.to(device, non_blocking=True)
        with torch.cuda.amp.autocast(enabled=amp_enabled):
            logits = model(images)
            loss = crit(logits, targets)

        probs = torch.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)

        y_true.extend(targets.detach().cpu().numpy())
        y_pred.extend(preds.detach().cpu().numpy())
        bs = images.size(0); total += bs; loss_meter += loss.item() * bs

        topk_idx = torch.topk(probs, k=k, dim=1).indices
        top5_correct += topk_idx.eq(targets.view(-1,1)).any(dim=1).sum().item()

    return finalize_metrics(y_true, y_pred, total, top5_correct, loss_meter, num_classes)

def _forward_maybe_chunked(model, x, max_splits=4):
    """
    Try one big forward; if OOM, clear cache and split into smaller chunks along dim 0.
    """
    try:
        return model(x)
    except RuntimeError as e:
        if "out of memory" not in str(e).lower():
            raise
        torch.cuda.empty_cache()
        splits = min(max_splits, max(1, math.ceil(x.size(0) / 256)))  # heuristic
        parts = torch.tensor_split(x, splits, dim=0)
        outs = []
        for p in parts:
            outs.append(model(p))
        return torch.cat(outs, dim=0)

@torch.inference_mode()
def evaluate_tta(model, loader, device, num_classes, img_size, mean, std, tta_level,
                 label_remap=None, use_amp=True):
    n_views, aug = make_tta_aug(tta_level, img_size, mean, std)
    base = A.Compose([
        A.LongestMaxSize(max_size=img_size),
        A.PadIfNeeded(min_height=img_size, min_width=img_size, border_mode=cv2.BORDER_REFLECT_101),
        A.Normalize(mean=tuple(mean), std=tuple(std)),
        ToTensorV2(),
    ])
    crit = torch.nn.CrossEntropyLoss()
    y_true, y_pred, total, loss_meter, top5_correct = [], [], 0, 0.0, 0
    k = min(5, num_classes)
    amp_enabled = (use_amp and device.startswith("cuda"))

    for imgs_np, targets in loader:
        bsz = len(imgs_np)
        # remap targets to ckpt label order (on CPU)
        if label_remap is not None:
            targets = label_remap[targets.cpu()].to(device, non_blocking=True)
        else:
            targets = targets.to(device, non_blocking=True)

        # Build N x CxHxW tensors for each view; stack into (n_views*N) x C x H x W
        view_batches = [torch.stack([base(image=img)["image"] for img in imgs_np], dim=0)]
        for _ in range(n_views - 1):
            view_batches.append(torch.stack([aug(image=img)["image"] for img in imgs_np], dim=0))
        batch_cat = torch.cat(view_batches, dim=0).to(device, non_blocking=True)  # (n_views*N) x C x H x W

        with torch.cuda.amp.autocast(enabled=amp_enabled):
            logits_cat = _forward_maybe_chunked(model, batch_cat)  # (n_views*N, num_classes)
            logits_avg = logits_cat.view(n_views, bsz, num_classes).mean(dim=0)
            loss = crit(logits_avg, targets)

        probs = torch.softmax(logits_avg, dim=1)
        preds = probs.argmax(dim=1)

        y_true.extend(targets.detach().cpu().numpy())
        y_pred.extend(preds.detach().cpu().numpy())
        total += bsz; loss_meter += loss.item() * bsz

        topk_idx = torch.topk(probs, k=k, dim=1).indices
        top5_correct += topk_idx.eq(targets.view(-1,1)).any(dim=1).sum().item()

    out = finalize_metrics(y_true, y_pred, total, top5_correct, loss_meter, num_classes)
    out.update({"tta_level": tta_level, "tta_views": n_views})
    return out

def finalize_metrics(y_true, y_pred, total, top5_correct, loss_meter, num_classes):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="macro", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="macro", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="macro", zero_division=0)
    cm   = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    report = classification_report(
        y_true, y_pred,
        labels=list(range(num_classes)), output_dict=True, zero_division=0
    )
    return {
        "acc": acc*100, "prec": prec*100, "rec": rec*100, "f1": f1*100,
        "top1": acc*100, "top5": 100.0 * (top5_correct / max(1, total)),
        "loss": loss_meter / max(1, total), "cm": cm, "report": report
    }

def main():
    t0 = time.time()
    assert os.path.isdir(TEST_DIR), f"TEST_DIR not found: {TEST_DIR}"
    os.makedirs(OUT_DIR, exist_ok=True)
    set_seed(RNG_SEED)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    ckpt_obj = torch.load(CKPT, map_location=device)
    sd, model_name, num_classes, class_names, img_size, mean, std = extract_model_info(ckpt_obj)

    # Build model and load weights safely
    model = timm.create_model(model_name, pretrained=False, num_classes=num_classes)
    missing, unexpected = model.load_state_dict(sd, strict=False)
    if missing or unexpected:
        print(f"[warn] load_state_dict: missing={len(missing)} unexpected={len(unexpected)}")
        if missing:   print("  missing keys (head mismatch?):", missing[:15], "..." if len(missing) > 15 else "")
        if unexpected:print("  unexpected keys:", unexpected[:15], "..." if len(unexpected) > 15 else "")
    model.to(device).eval()

    # Datasets / loaders with class-order remap
    use_tta = bool(TTA_LEVEL) and TTA_LEVEL in {"fast","strong","max"}

    if use_tta:
        test_ds = NumpyImageFolder(TEST_DIR)
    else:
        tfms = make_val_tfms(img_size, mean, std)
        test_ds = AlbumentationsImageFolder(TEST_DIR, tfms)

    # Class-order sanity handling
    ds_classes = test_ds.classes  # ImageFolder list (sorted by name)
    label_remap = build_label_remap(ds_classes, class_names)  # may be None
    if label_remap is None:
        assert (class_names is None) or (len(ds_classes) == len(class_names)), \
            f"Class count mismatch: dataset={len(ds_classes)} vs ckpt={len(class_names)}"
    else:
        identical = torch.equal(label_remap, torch.arange(len(ds_classes)))
        print(f"[info] class order identical to ckpt? {identical}. Remap built.")

    # DataLoader
    num_workers = min(8, os.cpu_count() or 2)
    pin = device == "cuda"
    if use_tta:
        test_loader = DataLoader(
            test_ds, batch_size=BATCH_TTA, shuffle=False,
            num_workers=num_workers, pin_memory=pin, collate_fn=collate_numpy
        )
        metrics = evaluate_tta(model, test_loader, device, num_classes, img_size, mean, std,
                               TTA_LEVEL, label_remap=label_remap, use_amp=True)
    else:
        test_loader = DataLoader(
            test_ds, batch_size=BATCH_NO_TTA, shuffle=False,
            num_workers=num_workers, pin_memory=pin
        )
        metrics = evaluate_no_tta(model, test_loader, device, num_classes,
                                  label_remap=label_remap, use_amp=True)

    # ===== SAVE RESULTS =====
    # 1) Human-readable summary
    with open(os.path.join(OUT_DIR, "metrics.txt"), "w") as f:
        f.write(f"Model: {model_name}\n")
        f.write(f"Device: {device}\n")
        f.write(f"Test Images: {len(test_ds)} | Classes: {num_classes}\n")
        f.write(f"ImgSize: {img_size} | Mean: {tuple(mean)} | Std: {tuple(std)}\n")
        if use_tta:
            f.write(f"TTA Level: {metrics['tta_level']} | Views: {metrics['tta_views']}\n")
        f.write("\n")
        f.write(f"Top-1 Acc: {metrics['top1']:.2f}%\n")
        f.write(f"Top-5 Acc: {metrics['top5']:.2f}%\n")
        f.write(f"Precision (macro): {metrics['prec']:.2f}%\n")
        f.write(f"Recall (macro): {metrics['rec']:.2f}%\n")
        f.write(f"F1 (macro): {metrics['f1']:.2f}%\n")
        f.write(f"CrossEntropy Loss: {metrics['loss']:.6f}\n")
        f.write(f"Elapsed: {time.time()-t0:.2f}s\n")

    # 2) JSON
    json.dump(metrics, open(os.path.join(OUT_DIR,"metrics.json"),"w"),
              indent=2, default=lambda x: x.tolist() if isinstance(x, np.ndarray) else x)

    # 3) CSVs
    class_names_out = class_names if class_names is not None else ds_classes
    pd.DataFrame(metrics["cm"], index=class_names_out, columns=class_names_out)\
      .to_csv(os.path.join(OUT_DIR,"confusion_matrix.csv"))
    rows=[]
    for i, cls in enumerate(class_names_out):
        stats = metrics["report"].get(str(i), {"precision":0.0,"recall":0.0,"f1-score":0.0,"support":0})
        rows.append({"class":cls, "precision":stats["precision"], "recall":stats["recall"],
                     "f1":stats["f1-score"], "support":stats["support"]})
    pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR,"per_class.csv"), index=False)

    # 4) Excel (summary + confusion matrix + per-class)
    with pd.ExcelWriter(os.path.join(OUT_DIR,"metrics.xlsx"), engine="openpyxl") as writer:
        pd.DataFrame([{
            "Top1_%":metrics["top1"], "Top5_%":metrics["top5"],
            "Macro_Precision_%":metrics["prec"], "Macro_Recall_%":metrics["rec"],
            "Macro_F1_%":metrics["f1"], "Loss":metrics["loss"],
            **({"TTA_Level": metrics.get("tta_level",""), "TTA_Views": metrics.get("tta_views","")} if use_tta else {})
        }]).to_excel(writer, index=False, sheet_name="summary")
        pd.DataFrame(metrics["cm"], index=class_names_out, columns=class_names_out)\
            .to_excel(writer, sheet_name="confusion_matrix")
        pd.DataFrame(rows).to_excel(writer, index=False, sheet_name="per_class")

    print(f"✅ Results saved to {OUT_DIR} | TTA: {'ON' if use_tta else 'OFF'} ({TTA_LEVEL if use_tta else '—'})")

if __name__ == "__main__":
    main()


/tmp/ipykernel_1258845/2661726633.py:341: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt_obj = torch.load(CKPT, map_location=device)


TypeError: cannot unpack non-iterable NoneType object

In [7]:
# === High-accuracy eval with auto-fix for missing class + strong TTA (Jupyter-friendly) ===
# deps: albumentations==2.x, timm, torch, torchvision, opencv-python, (optional) scipy

import os, csv
from pathlib import Path
from typing import List
import numpy as np
import torch, timm
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ------------------ CONFIG ------------------
WEIGHTS = "/home/gpu/PSL Isolated Signs Datasets/best_vit_psl.pth"
LABELED_DATA = "/home/gpu/PSL Isolated Signs Datasets/Validataion-2-Combined-cropped-dataset-"   # your eval folder
OUT_DIR = "/home/gpu/PSL Isolated Signs Datasets/eval_tta_out"
MISSING_CLASS_NAME = "ں"                      # the missing class you mentioned
PRESET = "strong"                             # "fast" | "strong" | "max"
FLIP_SAFE = True                              # you confirmed this

# ------------------ UTILITIES ------------------
def ensure_missing_class_folder(root: str, class_name: str):
    """Create an empty subfolder if it's missing so class index order matches training."""
    if not class_name:
        return
    target = Path(root) / class_name
    if not target.exists():
        target.mkdir(parents=True, exist_ok=True)
        print(f"[FIX] Created empty class folder: {target}")

def safe_imread_rgb(path: str):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None: raise RuntimeError(f"Failed to read: {path}")
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    elif img.ndim == 3:
        if img.shape[2] == 4:
            img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    else:
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

# --- TTA builders: pad BEFORE crop (avoids CropSizeError), then optional rotate/flip
def make_single_tta(img_size: int, scale: float, angle: float, do_flip: bool):
    ops = []
    max_side = max(1, int(round(img_size * scale)))
    ops.append(A.LongestMaxSize(max_size=max_side, interpolation=cv2.INTER_CUBIC))
    ops.append(A.PadIfNeeded(min_height=img_size, min_width=img_size,
                             border_mode=cv2.BORDER_REFLECT_101))
    if abs(angle) > 0:
        ops.append(A.Affine(rotate=angle, fit_output=False,
                            border_mode=cv2.BORDER_REFLECT_101))
    if do_flip:
        ops.append(A.HorizontalFlip(p=1.0))
    ops += [
        A.CenterCrop(img_size, img_size),
        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ]
    return A.Compose(ops)

def make_tta_list(img_size: int, preset: str, flip: bool):
    if preset == "fast":
        scales = [0.9, 1.0, 1.1]
        angles = [-8, 0, 8]
    elif preset == "strong":
        scales = [0.9, 1.0, 1.1, 1.2]
        angles = [-10, -5, 0, 5, 10]
    else:  # "max" (heaviest)
        scales = [0.85, 0.9, 0.95, 1.0, 1.1, 1.2]
        angles = [-12, -8, -4, 0, 4, 8, 12]
    tfs = []
    for s in scales:
        for a in angles:
            tfs.append(make_single_tta(img_size, s, a, False))
            if flip:
                tfs.append(make_single_tta(img_size, s, a, True))
    return tfs, scales, angles

# ------------------ EVAL (with mapping fallback) ------------------
@torch.no_grad()
def eval_with_tta_and_mapping(data_root: str, weights: str, out_dir: str,
                              preset="strong", flip=True, bs=64, workers=0):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # 1) Load ckpt+model
    ckpt = torch.load(weights, map_location="cpu")
    model_name = ckpt.get("model_name", "deit3_base_patch16_224")
    n_cls_ckpt = int(ckpt.get("classes", 39))
    img_size = int(ckpt.get("img_size", 224))
    state = ckpt["model"]

    model = timm.create_model(model_name, pretrained=False, num_classes=n_cls_ckpt)
    model.load_state_dict(state, strict=True)
    model.to(device).eval()

    # 2) Ensure missing class folder exists (so indices align)
    ensure_missing_class_folder(data_root, MISSING_CLASS_NAME)

    # 3) Dataset without transforms; we’ll apply tfm per TTA pass
    class ImageFolderRaw(ImageFolder):
        def __init__(self, root): super().__init__(root=root)
        def __getitem__(self, idx):
            path, y = self.samples[idx]
            return safe_imread_rgb(path), y

    base_ds = ImageFolderRaw(data_root)
    classes_ds = base_ds.classes
    n_cls_ds = len(classes_ds)
    N = len(base_ds)
    print(f"[INFO] ckpt_classes={n_cls_ckpt} | dataset_classes={n_cls_ds} | N={N}")

    # 4) Build TTA list
    tta_list, scales, angles = make_tta_list(img_size, preset, flip)
    print(f"[INFO] TTA variants: {len(tta_list)} | scales={scales} | angles={angles} | flip={flip}")

    # 5) Sum logits across TTAs
    sum_logits = torch.zeros((N, n_cls_ckpt), dtype=torch.float32, device=device)
    class _View(Dataset):
        def __init__(self, base, tfm): self.base, self.tfm = base, tfm
        def __len__(self): return len(self.base)
        def __getitem__(self, i):
            img, y = self.base[i]
            x = self.tfm(image=img)["image"]
            return x, y

    use_amp = torch.cuda.is_available()
    for t_idx, tfm in enumerate(tta_list, 1):
        view = _View(base_ds, tfm)
        dl = DataLoader(view, batch_size=bs, shuffle=False, num_workers=workers, pin_memory=True)
        ofs = 0
        for xb, yb in dl:
            xb = xb.to(device, non_blocking=True)
            if use_amp:
                with torch.amp.autocast("cuda"):
                    logits = model(xb)
            else:
                logits = model(xb)
            B = xb.size(0)
            sum_logits[ofs:ofs+B] += logits
            ofs += B
        print(f"TTA pass {t_idx}/{len(tta_list)} complete.")

    preds_raw = sum_logits.argmax(1).cpu().numpy()
    targets = np.array([y for _, y in base_ds.samples], dtype=np.int64)

    # 6) If classes still mismatch (shouldn’t after fix), do Hungarian mapping fallback
    if n_cls_ds != n_cls_ckpt:
        print("[WARN] class count mismatch persists; applying optimal label mapping.")
        cm_rect = np.zeros((n_cls_ds, n_cls_ckpt), dtype=np.int64)
        for t, p in zip(targets, preds_raw):
            cm_rect[t, p] += 1
        try:
            from scipy.optimize import linear_sum_assignment
            row_ind, col_ind = linear_sum_assignment(cm_rect.max() - cm_rect)
        except Exception:
            # Greedy fallback if scipy unavailable
            row_ind, col_ind = [], []
            used = set()
            for r in range(n_cls_ds):
                order = np.argsort(-cm_rect[r])
                c_pick = next((c for c in order if c not in used), int(order[0]))
                row_ind.append(r); col_ind.append(c_pick); used.add(c_pick)
            row_ind, col_ind = np.array(row_ind), np.array(col_ind)

        inv_map = {int(c): int(r) for r, c in zip(row_ind, col_ind)}
        preds_mapped = np.array([inv_map.get(int(p), -1) for p in preds_raw], dtype=np.int64)
        mask = preds_mapped >= 0
        acc_mapped = (preds_mapped[mask] == targets[mask]).mean() * 100.0 if mask.any() else 0.0
        acc_raw = (preds_raw == targets).mean() * 100.0
        os.makedirs(OUT_DIR, exist_ok=True)
        with open(os.path.join(out_dir, "VIT-STRONG-TTA-eval_tta_summary_mapped.txt"), "w", encoding="utf-8") as f:
            f.write(f"Raw acc: {acc_raw:.2f}%\nMapped acc: {acc_mapped:.2f}%\n")
        print(f"Raw acc: {acc_raw:.2f}%  |  Mapped acc: {acc_mapped:.2f}% → {out_dir}")
        return acc_mapped

    # 7) Normal accuracy (aligned classes)
    acc = (preds_raw == targets).mean() * 100.0
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, "VIT-STRONG-TTA-eval_tta_summary.txt"), "w", encoding="utf-8") as f:
        f.write(f"Accuracy: {acc:.2f}%\n")
        f.write(f"TTA preset={preset} | scales={scales} | angles={angles} | flip={flip}\n")
    print(f"Aligned classes. Acc: {acc:.2f}% → {out_dir}")
    return acc

# ------------------ RUN ------------------
os.makedirs(OUT_DIR, exist_ok=True)
acc_best = eval_with_tta_and_mapping(
    data_root=LABELED_DATA,
    weights=WEIGHTS,
    out_dir=OUT_DIR,
    preset=PRESET,         # "fast" | "strong" | "max"
    flip=FLIP_SAFE,        # True (safe for you)
    bs=1000,
    workers=20              # JupyterHub-stable; try 2–4 if you want more I/O speed
)
print("Final Accuracy:", f"{acc_best:.2f}%")


/tmp/ipykernel_282718/1386732719.py:91: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(weights, map_location="cpu")


[INFO] ckpt_classes=39 | dataset_classes=39 | N=5249
[INFO] TTA variants: 40 | scales=[0.9, 1.0, 1.1, 1.2] | angles=[-10, -5, 0, 5, 10] | flip=True
TTA pass 1/40 complete.
TTA pass 2/40 complete.
TTA pass 3/40 complete.
TTA pass 4/40 complete.
TTA pass 5/40 complete.
TTA pass 6/40 complete.
TTA pass 7/40 complete.
TTA pass 8/40 complete.
TTA pass 9/40 complete.
TTA pass 10/40 complete.
TTA pass 11/40 complete.
TTA pass 12/40 complete.
TTA pass 13/40 complete.
TTA pass 14/40 complete.
TTA pass 15/40 complete.
TTA pass 16/40 complete.
TTA pass 17/40 complete.
TTA pass 18/40 complete.
TTA pass 19/40 complete.
TTA pass 20/40 complete.
TTA pass 21/40 complete.
TTA pass 22/40 complete.
TTA pass 23/40 complete.
TTA pass 24/40 complete.
TTA pass 25/40 complete.
TTA pass 26/40 complete.
TTA pass 27/40 complete.
TTA pass 28/40 complete.
TTA pass 29/40 complete.
TTA pass 30/40 complete.
TTA pass 31/40 complete.
TTA pass 32/40 complete.
TTA pass 33/40 complete.
TTA pass 34/40 complete.
TTA pass 35

In [8]:
# === No-TTA inference with auto-fix for missing class (Jupyter-friendly) ===
# deps: albumentations==2.x, timm, torch, torchvision, opencv-python, (optional) scipy

import os, csv
from pathlib import Path
from typing import List
import numpy as np
import torch, timm
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ------------------ CONFIG ------------------
WEIGHTS = "/home/gpu/PSL Isolated Signs Datasets/best_vit_psl.pth"
LABELED_DATA = "/home/gpu/PSL Isolated Signs Datasets/Validataion-2-Combined-cropped-dataset-"   # your eval folder
OUT_DIR = "/home/gpu/PSL Isolated Signs Datasets/eval_tta_out"
MISSING_CLASS_NAME = "ں"                      # the missing class you mentioned
PRESET = "none"                               # <<< disable TTA
FLIP_SAFE = True                              # ignored when PRESET="none"

# ------------------ UTILITIES ------------------
def ensure_missing_class_folder(root: str, class_name: str):
    """Create an empty subfolder if it's missing so class index order matches training."""
    if not class_name:
        return
    target = Path(root) / class_name
    if not target.exists():
        target.mkdir(parents=True, exist_ok=True)
        print(f"[FIX] Created empty class folder: {target}")

def safe_imread_rgb(path: str):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if img is None: raise RuntimeError(f"Failed to read: {path}")
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    elif img.ndim == 3:
        if img.shape[2] == 4:
            img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    else:
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return img

# --- TTA/No-TTA builders ---
def make_single_tta(img_size: int, scale: float, angle: float, do_flip: bool):
    ops = []
    max_side = max(1, int(round(img_size * scale)))
    ops.append(A.LongestMaxSize(max_size=max_side, interpolation=cv2.INTER_CUBIC))
    ops.append(A.PadIfNeeded(min_height=img_size, min_width=img_size,
                             border_mode=cv2.BORDER_REFLECT_101))
    if abs(angle) > 0:
        ops.append(A.Affine(rotate=angle, fit_output=False,
                            border_mode=cv2.BORDER_REFLECT_101))
    if do_flip:
        ops.append(A.HorizontalFlip(p=1.0))
    ops += [
        A.CenterCrop(img_size, img_size),
        A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
        ToTensorV2(),
    ]
    return A.Compose(ops)

def make_tta_list(img_size: int, preset: str, flip: bool):
    # ---- NO-TTA path ----
    if preset == "none":
        tfm = A.Compose([
            A.LongestMaxSize(max_size=img_size, interpolation=cv2.INTER_CUBIC),
            A.PadIfNeeded(min_height=img_size, min_width=img_size,
                          border_mode=cv2.BORDER_REFLECT_101),
            A.CenterCrop(img_size, img_size),
            A.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5)),
            ToTensorV2(),
        ])
        return [tfm], [1.0], [0]

    # ---- existing TTA presets ----
    if preset == "fast":
        scales = [0.9, 1.0, 1.1]
        angles = [-8, 0, 8]
    elif preset == "strong":
        scales = [0.9, 1.0, 1.1, 1.2]
        angles = [-10, -5, 0, 5, 10]
    else:  # "max"
        scales = [0.85, 0.9, 0.95, 1.0, 1.1, 1.2]
        angles = [-12, -8, -4, 0, 4, 8, 12]
    tfs = []
    for s in scales:
        for a in angles:
            tfs.append(make_single_tta(img_size, s, a, False))
            if flip:
                tfs.append(make_single_tta(img_size, s, a, True))
    return tfs, scales, angles

# ------------------ EVAL (with mapping fallback) ------------------
@torch.no_grad()
def eval_with_tta_and_mapping(data_root: str, weights: str, out_dir: str,
                              preset="none", flip=True, bs=64, workers=0):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # 1) Load ckpt+model
    ckpt = torch.load(weights, map_location="cpu")
    model_name = ckpt.get("model_name", "deit3_base_patch16_224")
    n_cls_ckpt = int(ckpt.get("classes", 39))
    img_size = int(ckpt.get("img_size", 224))
    state = ckpt["model"]

    model = timm.create_model(model_name, pretrained=False, num_classes=n_cls_ckpt)
    model.load_state_dict(state, strict=True)
    model.to(device).eval()

    # 2) Ensure missing class folder exists (so indices align)
    ensure_missing_class_folder(data_root, MISSING_CLASS_NAME)

    # 3) Dataset without transforms; we’ll apply tfm per (TTA or single) pass
    class ImageFolderRaw(ImageFolder):
        def __init__(self, root): super().__init__(root=root)
        def __getitem__(self, idx):
            path, y = self.samples[idx]
            return safe_imread_rgb(path), y

    base_ds = ImageFolderRaw(data_root)
    classes_ds = base_ds.classes
    n_cls_ds = len(classes_ds)
    N = len(base_ds)
    print(f"[INFO] ckpt_classes={n_cls_ckpt} | dataset_classes={n_cls_ds} | N={N}")

    # 4) Build list (will be length=1 when preset='none')
    tta_list, scales, angles = make_tta_list(img_size, preset, flip)
    print(f"[INFO] Variants: {len(tta_list)} | scales={scales} | angles={angles} | flip={flip and preset!='none'}")

    # 5) Sum logits across variants (single pass for no-TTA)
    sum_logits = torch.zeros((N, n_cls_ckpt), dtype=torch.float32, device=device)
    class _View(Dataset):
        def __init__(self, base, tfm): self.base, self.tfm = base, tfm
        def __len__(self): return len(self.base)
        def __getitem__(self, i):
            img, y = self.base[i]
            x = self.tfm(image=img)["image"]
            return x, y

    use_amp = torch.cuda.is_available()
    for t_idx, tfm in enumerate(tta_list, 1):
        view = _View(base_ds, tfm)
        dl = DataLoader(view, batch_size=bs, shuffle=False, num_workers=workers, pin_memory=True)
        ofs = 0
        for xb, yb in dl:
            xb = xb.to(device, non_blocking=True)
            if use_amp:
                with torch.amp.autocast("cuda"):
                    logits = model(xb)
            else:
                logits = model(xb)
            B = xb.size(0)
            sum_logits[ofs:ofs+B] += logits
            ofs += B
        print(f"Pass {t_idx}/{len(tta_list)} complete.")

    preds_raw = sum_logits.argmax(1).cpu().numpy()
    targets = np.array([y for _, y in base_ds.samples], dtype=np.int64)

    # 6) If classes mismatch, do Hungarian mapping fallback
    if n_cls_ds != n_cls_ckpt:
        print("[WARN] class count mismatch persists; applying optimal label mapping.")
        cm_rect = np.zeros((n_cls_ds, n_cls_ckpt), dtype=np.int64)
        for t, p in zip(targets, preds_raw):
            cm_rect[t, p] += 1
        try:
            from scipy.optimize import linear_sum_assignment
            row_ind, col_ind = linear_sum_assignment(cm_rect.max() - cm_rect)
        except Exception:
            # Greedy fallback if scipy unavailable
            row_ind, col_ind = [], []
            used = set()
            for r in range(n_cls_ds):
                order = np.argsort(-cm_rect[r])
                c_pick = next((c for c in order if c not in used), int(order[0]))
                row_ind.append(r); col_ind.append(c_pick); used.add(c_pick)
            row_ind, col_ind = np.array(row_ind), np.array(col_ind)

        inv_map = {int(c): int(r) for r, c in zip(row_ind, col_ind)}
        preds_mapped = np.array([inv_map.get(int(p), -1) for p in preds_raw], dtype=np.int64)
        mask = preds_mapped >= 0
        acc_mapped = (preds_mapped[mask] == targets[mask]).mean() * 100.0 if mask.any() else 0.0
        os.makedirs(OUT_DIR, exist_ok=True)
        with open(os.path.join(out_dir, "VIT-NO-TTA-eval_summary_mapped.txt"), "w", encoding="utf-8") as f:
            f.write(f"Raw acc: {((preds_raw == targets).mean() * 100.0):.2f}%\nMapped acc: {acc_mapped:.2f}%\n")
        print(f"Raw acc: {((preds_raw == targets).mean() * 100.0):.2f}%  |  Mapped acc: {acc_mapped:.2f}% → {out_dir}")
        return acc_mapped

    # 7) Normal accuracy (aligned classes)
    acc = (preds_raw == targets).mean() * 100.0
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, "VIT-NO-TTA-eval_summary.txt"), "w", encoding="utf-8") as f:
        f.write(f"Accuracy: {acc:.2f}%\n")
        f.write(f"preset={preset} (no TTA)\n")
    print(f"Aligned classes. Acc: {acc:.2f}% → {out_dir}")
    return acc

# ------------------ RUN ------------------
os.makedirs(OUT_DIR, exist_ok=True)
acc_best = eval_with_tta_and_mapping(
    data_root=LABELED_DATA,
    weights=WEIGHTS,
    out_dir=OUT_DIR,
    preset=PRESET,         # "none" → no TTA
    flip=FLIP_SAFE,        # ignored when PRESET="none"
    bs=1000,
    workers=20
)
print("Final Accuracy:", f"{acc_best:.2f}%")


/tmp/ipykernel_282718/1199382757.py:104: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(weights, map_location="cpu")


[INFO] ckpt_classes=39 | dataset_classes=39 | N=5249
[INFO] Variants: 1 | scales=[1.0] | angles=[0] | flip=False
Pass 1/1 complete.
Aligned classes. Acc: 94.34% → /home/gpu/PSL Isolated Signs Datasets/eval_tta_out
Final Accuracy: 94.34%
